# Leave-one-cultivar-out transfer experiments

The notebook records the experiment configuration, analysis procedure and saved outputs for this study.


## 1 — Configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, random, time, pickle, traceback, copy, threading
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

V4 = Path(os.getenv('DATEPALM_V4_ROOT', 'external_data')).expanduser().resolve()
STAGE = Path(os.getenv('DATEPALM_V51_ROOT', 'external_data')).expanduser().resolve()
OUT = Path(os.getenv('DATEPALM_PROBE_V3_OUT', str(STAGE/'additional_evidence_20261024'/'independent_training_probes_resumable_v3'))).expanduser().resolve()
MANIFEST = V4/'data/source_split.csv'
IMAGE_MAP = STAGE/'verified_v4_image_path_map.csv'
RUN_TRAINING = True                 # explicit manual opt-in after successful preflight
USE_CUDA = True                      # True = CUDA, False = CPU
RUN_CLASS_HOLDOUT = False            # optional three class-holdout experiments
VERIFY_IMAGE_HASHES = True          # persistent SQLite hash cache; only changed files are rehashed
SEEDS = (11, 23, 37)
VARIETIES = ('Aseel', 'Dandhi', 'Karblian', 'Kupro')
SIZES = ('Small', 'Medium', 'Large')
EPOCHS = 12
PATIENCE = 4
LR = 0.0002
WEIGHT_DECAY = 0.001
BATCH_SIZE = 32
MONITOR_EVERY_N_BATCHES = 1
CHECKPOINT_EVERY_N_BATCHES = 1       # 1 = strongest minibatch-level recovery, disk I/O tradeoff
INFERENCE_BATCH_SIZE = 32
SOURCE_VERSION = 'DatePalm-Transfer-Probes-v3.1-minibatch-resume'
MAX_NEW_RUNS = None                  # None = all; 1 = one newly started experiment per invocation

assert OUT != V4 and OUT != STAGE
assert V4 not in OUT.parents and STAGE not in OUT.parents or (STAGE in OUT.parents and OUT != STAGE)
assert V4 not in OUT.parents and OUT not in V4.parents
OUT.mkdir(parents=True, exist_ok=True)

def utc_now(): return datetime.now(timezone.utc).isoformat(timespec='seconds')
def json_atomic(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.writing')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, sort_keys=True, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def log_event(event, **details):
    obj = {'time_utc':utc_now(),'event':event, **details}
    with open(OUT/'events.jsonl','a',encoding='utf-8') as f:
        f.write(json.dumps(obj,default=str)+'\n'); f.flush(); os.fsync(f.fileno())
    print(f'[{obj["time_utc"]}] {event}: ' + ' | '.join(f'{k}={v}' for k,v in details.items()),flush=True)

print('V3 output:',OUT)
print('Original V4:',V4,'\nImage map:',IMAGE_MAP)
print('TRAINING:',RUN_TRAINING,'CUDA requested:',USE_CUDA)
print('Previous partial V2 outputs are not overwritten.')

## 2 — Dataset identity + resumable image-hash verification


In [ ]:
import sqlite3
assert MANIFEST.is_file(), f'V4 manifest unavailable: {MANIFEST}'
assert IMAGE_MAP.is_file(), f'Verified image map unavailable: {IMAGE_MAP}'
M = pd.read_csv(MANIFEST,dtype=str,keep_default_na=False)
F = pd.read_csv(IMAGE_MAP,dtype=str,keep_default_na=False)
for col in ('source_id','source_group','split','eligibility','native_size_label','variety','file_sha256'):
    assert col in M, f'Manifest lacks {col}'
for col in ('source_id','verified_image_path','file_sha256'):
    assert col in F, f'Image-map lacks {col}'
assert M.source_id.is_unique and F.source_id.is_unique
assert M.groupby('source_group').split.nunique().max() == 1, 'Source-group crosses frozen partitions'
assert M.groupby('file_sha256').split.nunique().max() == 1, 'Exact SHA across partitions'
M = M.loc[M.eligibility.eq('eligible')].copy()
assert len(M)==5828 and M.split.value_counts().to_dict()=={'train':3603,'development':835,'locked_test':834,'calibration':556}
D=M.loc[M.split.isin(['train','development'])].copy()
D=D.merge(F[['source_id','file_sha256','verified_image_path']],on='source_id',validate='one_to_one',suffixes=('','_map'))
assert D.file_sha256.str.lower().eq(D.file_sha256_map.str.lower()).all(), 'Identity mismatch between manifest and verified map'
assert D.verified_image_path.map(lambda x:Path(x).is_file()).all(), 'Missing source photographs'
assert D.groupby('source_group').split.nunique().max()==1

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

hash_db=OUT/'verified_image_hashes.sqlite3'
with sqlite3.connect(hash_db) as conn:
    conn.execute('CREATE TABLE IF NOT EXISTS valid (path TEXT PRIMARY KEY, size INTEGER, mtime_ns INTEGER, expected TEXT NOT NULL, checked_utc TEXT NOT NULL)')
    conn.commit()
    previous={p:(sz,mt,ex) for p,sz,mt,ex in conn.execute('SELECT path,size,mtime_ns,expected FROM valid')}
    seen=set(); reused=0; hashed=0
    try:
        for r in tqdm(D[['verified_image_path','file_sha256']].itertuples(index=False),total=len(D),desc='Verify source SHA-256',unit='image',mininterval=1):
            path=str(r.verified_image_path); expected=str(r.file_sha256).lower()
            if path in seen: continue
            seen.add(path)
            st=Path(path).stat()
            if VERIFY_IMAGE_HASHES:
                if previous.get(path)==(st.st_size,st.st_mtime_ns,expected):
                    reused+=1
                else:
                    actual=sha_file(path)
                    if actual.lower()!=expected: raise RuntimeError(f'Original-image SHA-256 mismatch: {path}')
                    conn.execute('INSERT OR REPLACE INTO valid VALUES (?,?,?,?,?)',(path,st.st_size,st.st_mtime_ns,expected,utc_now()))
                    conn.commit()  # durable even if notebook is interrupted right now
                    hashed+=1
            else:
                reused+=1
    except BaseException:
        conn.commit(); raise
print(f'Integrity cache: already verified={reused}, newly hashed={hashed}, unique paths={len(seen)}',flush=True)
print('Dataset counts:',len(D),D.split.value_counts().to_dict())
log_event('SOURCE_AUDIT_OK',images=len(D),new_hashes=hashed,cached_hashes=reused)

## 3 — Frozen split-safe fold generation and run identity


In [ ]:
def stable_half(group):
    return int(hashlib.sha256(('datepalm_secondary_holdout_20261024/'+str(group)).encode()).hexdigest()[:8],16)%2

def make_fold(df,heldout,dimension):
    assert dimension in ('variety','native_size_label')
    forbidden=set(df.loc[df[dimension].eq(heldout),'source_group'])
    fit=df.loc[df.split.eq('train') & ~df.source_group.isin(forbidden)].copy()
    known=df.loc[df.split.eq('development') & ~df.source_group.isin(forbidden)].copy()
    evaluation=df.loc[df.split.eq('development') & df[dimension].eq(heldout)].copy()
    known['half']=known.source_group.map(stable_half)
    select=known.loc[known.half.eq(0)].drop(columns='half')
    calibrate=known.loc[known.half.eq(1)].drop(columns='half')
    parts=[fit,select,calibrate,evaluation]
    assert all(len(x) for x in parts),f'Empty fold for {dimension}={heldout}'
    ids=[set(x.source_group) for x in parts]
    assert all(not(ids[i] & ids[j]) for i in range(4) for j in range(i+1,4))
    assert heldout not in set(fit[dimension]) | set(select[dimension]) | set(calibrate[dimension])
    assert set(evaluation[dimension])=={heldout}
    assert all(set(x.split).issubset({'train','development'}) for x in parts)
    return tuple(parts)

folds={('LOVO',v):make_fold(D,v,'variety') for v in VARIETIES}
if RUN_CLASS_HOLDOUT:
    folds.update({('CLASS_HOLDOUT',x):make_fold(D,x,'native_size_label') for x in SIZES})
inventory=[]
for (mode,heldout),(fit,sel,cal,ev) in folds.items():
    for name,frame in [('train',fit),('selection',sel),('calibration',cal),('evaluation',ev)]:
        inventory.append({'mode':mode,'holdout':heldout,'part':name,'n_images':len(frame),'n_groups':frame.source_group.nunique(), 'size_counts':frame.native_size_label.value_counts().to_dict()})
display(pd.DataFrame(inventory)[['mode','holdout','part','n_images','n_groups']])
pd.DataFrame(inventory).to_csv(OUT/'FOLD_INVENTORY.csv',index=False)
DATA_INPUTS={'manifest_sha256':sha_file(MANIFEST),'image_map_sha256':sha_file(IMAGE_MAP)}
CONFIG={'version':SOURCE_VERSION,'epochs':EPOCHS,'patience':PATIENCE,'lr':LR,'wd':WEIGHT_DECAY,'batch_size':BATCH_SIZE,'inference_batch_size':INFERENCE_BATCH_SIZE,'sizes':SIZES,'varieties':VARIETIES,'seeds':SEEDS,'fold_kind':'source_disjoint_LOVO_with_separate_development_sel_cal','preprocess':'EXIF BICUBIC letterbox gray128 -> bilinear224 -> ImageNet V1 normalization','training':'ResNet18 ImageNetV1 layer4+fc AdamW epoch macro-F1 selection'}
base_fp=hashlib.sha256(json.dumps({'inputs':DATA_INPUTS,'config':CONFIG},sort_keys=True,default=str).encode()).hexdigest()
json_atomic(OUT/'EXPERIMENT_IDENTITY.json',{'data':DATA_INPUTS,'config':CONFIG,'identity_sha256':base_fp})
print('Run identity:',base_fp[:16])

## 4 — Pytorch, explicit pretrained-weight caching and determinism


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from PIL import Image, ImageOps
from sklearn.metrics import f1_score,accuracy_score,roc_auc_score
print('torch',torch.__version__,'CUDA available:',torch.cuda.is_available())
PRE=transforms.Compose([transforms.Resize((224,224),interpolation=transforms.InterpolationMode.BILINEAR),transforms.ToTensor(),transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
def image_tensor(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im.thumbnail((256,256),Image.Resampling.BICUBIC)
        canvas=Image.new('RGB',(256,256),(128,128,128))
        canvas.paste(im,((256-im.width)//2,(256-im.height)//2))
        return PRE(canvas)

class Fruits(Dataset):
    def __init__(self,frame,classes):self.frame=frame.reset_index(drop=True);self.idx={v:i for i,v in enumerate(classes)}
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[int(i)]
        return image_tensor(r.verified_image_path),self.idx.get(r.native_size_label,-1)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def rng_pack():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
def rng_restore(p):
    random.setstate(p['python']);np.random.set_state(p['numpy']);torch.set_rng_state(p['torch'])
    if torch.cuda.is_available() and p.get('cuda') is not None:torch.cuda.set_rng_state_all(p['cuda'])

def atomic_torch_save(obj,target):
    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)
    tmp=target.with_suffix('.writing')
    with open(tmp,'wb') as f:
        torch.save(obj,f); f.flush();os.fsync(f.fileno())
    os.replace(tmp,target)

def load_private_ckpt(target,device):
    # These checkpoints are written only by THIS notebook. Never load untrusted PyTorch pickle files.
    return torch.load(target,map_location=device,weights_only=False)

def pretrained_state():
    log_event('PRETRAINED_WEIGHTS_CHECK',model='ResNet18 ImageNet1K_V1')
    m=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    state={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
    print('ImageNet state loaded/cached. Number of tensors:',len(state),flush=True)
    del m
    return state

def new_model(classes,initial):
    m=models.resnet18(weights=None)
    m.load_state_dict(initial)
    m.fc=nn.Linear(m.fc.in_features,len(classes))
    for name,p in m.named_parameters():p.requires_grad_(name.startswith('layer4.') or name.startswith('fc.'))
    return m

## 5 — Checkpointed training procedure (one checkpoint after every minibatch)


In [ ]:
def run_status(folder,phase,**kw):
    data={'updated_utc':utc_now(),'phase':phase,**kw}
    json_atomic(folder/'STATE.json',data)
    json_atomic(OUT/'RUN_STATUS.json',{'folder':str(folder),**data})

class LiveHeartbeat:
    """Print and persist a heartbeat even when a forward/backward pass is slow."""
    def __init__(self,folder,run,interval=15):
        self.folder=folder;self.run=run;self.interval=interval
        self.phase='starting';self.detail='';self.epoch=None;self.batch=None
        self._stop=threading.Event();self._thread=None
        self.started=time.monotonic()
    def start(self):
        def worker():
            while not self._stop.wait(self.interval):
                payload={'updated_utc':utc_now(),'phase':self.phase,'run':self.run,
                         'detail':self.detail,'epoch':self.epoch,'batch':self.batch,
                         'elapsed_seconds':round(time.monotonic()-self.started,1)}
                try:
                    json_atomic(self.folder/'HEARTBEAT.json',payload)
                    print(f'[{payload["updated_utc"]}] HEARTBEAT {self.run}: {self.phase} | {self.detail} | elapsed {payload["elapsed_seconds"]}s',flush=True)
                except Exception as exc:
                    print('Heartbeat write warning:',repr(exc),flush=True)
        self._thread=threading.Thread(target=worker,daemon=True,name='DatePalmV3Heartbeat')
        self._thread.start()
        return self
    def update(self,phase,detail='',epoch=None,batch=None):
        self.phase=phase;self.detail=detail;self.epoch=epoch;self.batch=batch
    def stop(self):
        self._stop.set()
        if self._thread is not None:self._thread.join(timeout=1)
        json_atomic(self.folder/'HEARTBEAT.json',{'updated_utc':utc_now(),'run':self.run,'phase':'heartbeat_stopped'})

def run_fp(mode,heldout,seed,parts):
    ids={key:hashlib.sha256(('\n'.join(frame.source_id.astype(str))).encode()).hexdigest() for key,frame in zip(('train','selection','calibration','evaluation'),parts)}
    return hashlib.sha256(json.dumps({'base':base_fp,'mode':mode,'holdout':heldout,'seed':seed,'IDs':ids},sort_keys=True).encode()).hexdigest()

def cached_inference(model,frame,classes,device,folder,label,fp):
    cache=folder/'inference_cache'/label;cache.mkdir(parents=True,exist_ok=True)
    ds=Fruits(frame,classes); n=(len(ds)+INFERENCE_BATCH_SIZE-1)//INFERENCE_BATCH_SIZE
    probs=[];truth=[]
    model.eval()
    with torch.no_grad():
        for bi in tqdm(range(n),desc=f'Inference {label}',unit='batch',mininterval=1,leave=False):
            fp_file=cache/f'batch_{bi:05d}.npz'
            lo=bi*INFERENCE_BATCH_SIZE;hi=min(len(ds),lo+INFERENCE_BATCH_SIZE)
            if fp_file.is_file():
                arr=np.load(fp_file,allow_pickle=False)
                valid=str(arr['fingerprint'])==fp and int(arr['begin'])==lo and int(arr['end'])==hi
                if not valid:raise RuntimeError(f'Stale inference cache detected: {fp_file}')
                pp=arr['probabilities'];yy=arr['labels']
            else:
                xb=[];yb=[]
                for j in range(lo,hi):
                    x,y=ds[j];xb.append(x);yb.append(y)
                logits=model(torch.stack(xb).to(device))
                pp=torch.softmax(logits,dim=1).cpu().numpy();yy=np.asarray(yb,dtype=np.int64)
                tmp=cache/f'batch_{bi:05d}.writing.npz'
                np.savez_compressed(tmp,probabilities=pp,labels=yy,begin=lo,end=hi,fingerprint=fp)
                os.replace(tmp,fp_file)
            probs.append(pp);truth.append(yy)
            if bi%MONITOR_EVERY_N_BATCHES==0 or bi==n-1:
                run_status(folder,'inference',label=label,batches_done=bi+1,batches_total=n)
                print(f'[{utc_now()}] inference {label}: {bi+1}/{n} batches cached',flush=True)
    return np.concatenate(probs),np.concatenate(truth)

def training_state(model,opt,epoch,next_batch,accum_loss,accum_n,best,best_epoch,stale,history,fp):
    return {'fingerprint':fp,'model':model.state_dict(),'optimizer':opt.state_dict(),
            'epoch':epoch,'next_batch':next_batch,'loss_sum':accum_loss,'loss_count':accum_n,
            'best':best,'best_epoch':best_epoch,'stale':stale,'history':history,'rng':rng_pack(),
            'saved_utc':utc_now()}

def run_one(mode,heldout,parts,seed,init_state,device):
    fit,select,calibrate,evaluate=parts
    classes=tuple(c for c in SIZES if not(mode=='CLASS_HOLDOUT' and c==heldout))
    for frame in (fit,select,calibrate):assert frame.native_size_label.isin(classes).all()
    name=f'{mode}_{heldout}_seed{seed}'.replace(' ','_')
    folder=OUT/name;folder.mkdir(exist_ok=True)
    fp=run_fp(mode,heldout,seed,parts)
    final=folder/'RESULT.json'; trainfile=folder/'TRAIN_STATE.pt';bestfile=folder/'BEST_MODEL.pt'
    if final.exists():
        record=json.loads(final.read_text())
        assert record.get('status')=='COMPLETED' and record.get('fingerprint')==fp
        assert (folder/'HELDOUT_PREDICTIONS.csv').exists() and bestfile.exists()
        log_event('RUN_CACHE_HIT',run=name)
        return record
    seed_all(seed)
    model=new_model(classes,init_state).to(device)
    opt=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=LR,weight_decay=WEIGHT_DECAY)
    objective=nn.CrossEntropyLoss()
    best=-float('inf');best_epoch=0;stale=0;history=[];epoch=1;next_batch=0;loss_sum=0.;loss_count=0
    if trainfile.exists():
        z=load_private_ckpt(trainfile,device)
        assert z.get('fingerprint')==fp, f'Checkpoint fingerprint mismatch for {name}: refuse to continue'
        model.load_state_dict(z['model']);opt.load_state_dict(z['optimizer'])
        epoch=int(z['epoch']);next_batch=int(z['next_batch']);loss_sum=float(z['loss_sum']);loss_count=int(z['loss_count'])
        best=float(z['best']);best_epoch=int(z['best_epoch']);stale=int(z['stale']);history=list(z['history']);rng_restore(z['rng'])
        log_event('RESUME',run=name,epoch=epoch,next_batch=next_batch,previous_epochs=len(history))
    else:
        log_event('START_NEW_RUN',run=name,fit_images=len(fit),evaluation_images=len(evaluate),device=str(device))
    ds=Fruits(fit,classes)
    n_batches=(len(ds)+BATCH_SIZE-1)//BATCH_SIZE
    assert next_batch<=n_batches
    start_clock=time.monotonic()
    heartbeat=LiveHeartbeat(folder,name).start()
    try:
        while epoch<=EPOCHS and stale<PATIENCE:
            heartbeat.update('training',f'epoch {epoch} / {EPOCHS}, preparing minibatches',epoch=epoch,batch=next_batch)
            model.train()
            # Pretrained running statistics remain fixed in layers 1-3.
            model.bn1.eval();model.layer1.eval();model.layer2.eval();model.layer3.eval()
            perm=torch.randperm(len(ds),generator=torch.Generator().manual_seed(seed*100000+epoch)).tolist()
            with tqdm(total=n_batches,initial=next_batch,desc=f'{name} epoch {epoch}/{EPOCHS}',unit='batch',mininterval=1) as bar:
                for batch in range(next_batch,n_batches):
                    heartbeat.update('training',f'epoch {epoch}/{EPOCHS}, batch {batch+1}/{n_batches}',epoch=epoch,batch=batch+1)
                    indices=perm[batch*BATCH_SIZE:(batch+1)*BATCH_SIZE]
                    xx=[];yy=[]
                    for ix in indices:
                        x,y=ds[ix];xx.append(x);yy.append(y)
                    x=torch.stack(xx).to(device);y=torch.as_tensor(yy,dtype=torch.long,device=device)
                    opt.zero_grad(set_to_none=True)
                    loss=objective(model(x),y);loss.backward();opt.step()
                    loss_sum+=float(loss.item());loss_count+=1
                    next_batch=batch+1
                    if next_batch%CHECKPOINT_EVERY_N_BATCHES==0 or next_batch==n_batches:
                        atomic_torch_save(training_state(model,opt,epoch,next_batch,loss_sum,loss_count,best,best_epoch,stale,history,fp),trainfile)
                    if next_batch%MONITOR_EVERY_N_BATCHES==0 or next_batch==n_batches:
                        run_status(folder,'training',run=name,epoch=epoch,epochs=EPOCHS,batches_done=next_batch,batches_total=n_batches,mean_loss=round(loss_sum/loss_count,5),last_checkpoint=str(trainfile))
                        print(f'[{utc_now()}] {name} epoch {epoch} batch {next_batch}/{n_batches}, mean loss {loss_sum/loss_count:.5f}, elapsed {time.monotonic()-start_clock:.0f}s',flush=True)
                    bar.update(1);bar.set_postfix(loss=f'{loss_sum/loss_count:.4f}')
            # If interrupted during selection inference, its batches are reusable on restart.
            heartbeat.update('selection_inference',f'epoch {epoch} validation')
            fingerprint=f'{fp}_selection_epoch_{epoch}'
            sp,sy=cached_inference(model,select,classes,device,folder,f'selection_epoch_{epoch:03d}',fingerprint)
            sf1=float(f1_score(sy,sp.argmax(axis=1),labels=list(range(len(classes))),average='macro',zero_division=0))
            mean_loss=loss_sum/max(loss_count,1)
            history.append({'epoch':epoch,'training_loss':mean_loss,'selection_macro_F1':sf1,'elapsed_run_seconds':round(time.monotonic()-start_clock,2)})
            if sf1>best+1e-8:
                best=sf1;best_epoch=epoch;stale=0
                atomic_torch_save({'fingerprint':fp,'model':model.state_dict(),'best_epoch':epoch,'selection_macro_F1':sf1,'seed':seed,'classes':classes},bestfile)
                log_event('BEST_MODEL_COMMITTED',run=name,epoch=epoch,selection_macro_F1=round(sf1,5))
            else:stale+=1
            pd.DataFrame(history).to_csv(folder/'TRAINING_HISTORY.csv',index=False)
            log_event('EPOCH_END',run=name,epoch=epoch,loss=round(mean_loss,5),selection_macro_F1=round(sf1,5),best_epoch=best_epoch,stale=stale)
            epoch+=1;next_batch=0;loss_sum=0.;loss_count=0
            atomic_torch_save(training_state(model,opt,epoch,next_batch,loss_sum,loss_count,best,best_epoch,stale,history,fp),trainfile)
        assert bestfile.exists(),f'No best checkpoint for {name}'
        heartbeat.update('final_inference','loading best model')
        ck=load_private_ckpt(bestfile,device);assert ck['fingerprint']==fp
        model.load_state_dict(ck['model']);model.eval()
        cal_p,cal_y=cached_inference(model,calibrate,classes,device,folder,'final_calibration',fp+'_best_epoch_'+str(best_epoch))
        ev_p,ev_y=cached_inference(model,evaluate,classes,device,folder,'final_evaluation',fp+'_best_epoch_'+str(best_epoch))
        cal_unc=1-cal_p.max(axis=1);ev_unc=1-ev_p.max(axis=1)
        threshold=float(np.quantile(cal_unc,.90,method='higher'))
        referred=ev_unc>threshold
        report={'status':'COMPLETED','fingerprint':fp,'mode':mode,'excluded_group':heldout,'seed':seed,
                'n_fit':len(fit),'n_selection':len(select),'n_calibration':len(calibrate),'n_evaluation':len(evaluate),
                'best_epoch':best_epoch,'selection_macro_F1':best,'calibration_threshold_90pct':threshold,
                'heldout_referral_fraction':float(referred.mean()),'heldout_mean_max_probability':float(ev_p.max(axis=1).mean()),
                'evidence_type':'RETROSPECTIVE_TRANSFER_PROBE_NOT_EXTERNAL_VALIDATION','source_version':SOURCE_VERSION}
        if mode=='LOVO':
            pred=ev_p.argmax(axis=1)
            report['heldout_accuracy']=float(accuracy_score(ev_y,pred))
            report['heldout_macro_F1']=float(f1_score(ev_y,pred,labels=[0,1,2],average='macro',zero_division=0))
            groups=evaluate.source_group.astype(str).to_numpy();gids=np.unique(groups)
            indices=[np.flatnonzero(groups==g) for g in gids]
            brng=np.random.default_rng(seed+20261024)
            boot=[]
            for _ in tqdm(range(1000),desc=f'Bootstrap {name}',unit='replicate',leave=False):
                picks=brng.integers(len(indices),size=len(indices))
                subset=np.concatenate([indices[j] for j in picks])
                boot.append((accuracy_score(ev_y[subset],pred[subset]),f1_score(ev_y[subset],pred[subset],labels=[0,1,2],average='macro',zero_division=0)))
            bounds=np.percentile(np.asarray(boot),[2.5,97.5],axis=0)
            report['accuracy_CI95_source_group']=bounds[:,0].tolist();report['macro_F1_CI95_source_group']=bounds[:,1].tolist()
        else:
            report['withheld_class_OOD_AUROC']=float(roc_auc_score(np.r_[np.zeros(len(cal_unc)),np.ones(len(ev_unc))],np.r_[cal_unc,ev_unc]))
        result=evaluate[['source_id','source_group','variety','native_size_label','split']].copy()
        result['predicted_label']=[classes[i] for i in ev_p.argmax(axis=1)]
        result['max_probability']=ev_p.max(axis=1);result['uncertainty']=ev_unc;result['referred']=referred
        for j,cl in enumerate(classes):result['prob_'+cl]=ev_p[:,j]
        temp=folder/'HELDOUT_PREDICTIONS.writing.csv';result.to_csv(temp,index=False);os.replace(temp,folder/'HELDOUT_PREDICTIONS.csv')
        json_atomic(final,report)
        run_status(folder,'COMPLETED',run=name,best_epoch=best_epoch,report=str(final))
        log_event('RUN_COMPLETED',run=name,best_epoch=best_epoch,heldout_n=len(evaluate))
        return report
    except BaseException as error:
        # The most recently committed full minibatch checkpoint remains valid.
        run_status(folder,'INTERRUPTED_OR_FAILED',run=name,epoch=epoch,next_batch=next_batch,checkpoint=str(trainfile),error=str(error))
        log_event('RUN_INTERRUPTED',run=name,epoch=epoch,next_batch=next_batch,error=repr(error))
        raise
    finally:
        heartbeat.stop()

## 6 — Execute and resume experiments


In [ ]:
if not RUN_TRAINING:
    print('PREFLIGHT COMPLETE. Training NOT started. Set RUN_TRAINING=True when you want to start/resume training.',flush=True)
    json_atomic(OUT/'RUN_STATUS.json',{'phase':'PREFLIGHT_ONLY','updated_utc':utc_now(),'n_folds':len(folds)})
else:
    if USE_CUDA:
        assert torch.cuda.is_available(), 'CUDA requested but unavailable. Stop; resolve CUDA or explicitly select CPU.'
        device=torch.device('cuda')
    else:device=torch.device('cpu')
    log_event('EXECUTION_STARTED',device=str(device),experiments=len(folds)*len(SEEDS))
    initial=pretrained_state()
    records=[];new_runs=0
    for (mode,heldout),parts in folds.items():
        for seed in SEEDS:
            name=f'{mode}_{heldout}_seed{seed}'.replace(' ','_')
            if MAX_NEW_RUNS is not None and not (OUT/name/'RESULT.json').exists() and new_runs>=MAX_NEW_RUNS:
                log_event('INVOCATION_LIMIT_REACHED',remaining='Run All again to continue')
                break
            if not (OUT/name/'RESULT.json').exists():new_runs+=1
            records.append(run_one(mode,heldout,parts,seed,initial,device))
            pd.DataFrame(records).to_csv(OUT/'ALL_EXPERIMENT_RESULTS_PARTIAL.csv',index=False)
        else:continue
        break
    total=len(folds)*len(SEEDS)
    completed=sum((OUT/f'{mode}_{holdout}_seed{seed}'/'RESULT.json').exists() for mode,holdout in folds for seed in SEEDS)
    if completed==total:
        gathered=[json.loads((OUT/f'{mode}_{holdout}_seed{seed}'/'RESULT.json').read_text()) for mode,holdout in folds for seed in SEEDS]
        pd.DataFrame(gathered).to_csv(OUT/'ALL_EXPERIMENT_RESULTS.csv',index=False)
        json_atomic(OUT/'RESULTS_COMPLETE.json',{'status':'COMPLETED','n_completed':completed,'n_expected':total,'completed_utc':utc_now(),'independent_external_validation':False})
        log_event('ALL_COMPLETE',n_completed=completed,results=str(OUT/'ALL_EXPERIMENT_RESULTS.csv'))
    else:log_event('PARTIAL_COMPLETION',n_completed=completed,n_expected=total)
    print('Completed',completed,'/',total,'runs. All completed runs remain cached.',flush=True)

## 7 — Saved outputs and run metadata
